# Airbnb Pricing & Market Intelligence

## Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
dim_listing = pd.read_csv(
    "../data/processed/dim_listing.csv"
)

dim_host = pd.read_csv(
    "../data/processed/dim_host.csv",
    parse_dates=["host_since"],
    low_memory=False
)

fact_reviews = pd.read_csv(
    "../data/processed/fact_reviews.csv",
    parse_dates=["date"]
)

print("Tables Loaded Successfully")

Tables Loaded Successfully


In [3]:
print(dim_listing.columns.tolist())

['listing_id', 'host_id', 'city', 'neighbourhood', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value']


### Dataset Overview

In [4]:
overview = pd.DataFrame({
    "Metric": [
        "Total Listings",
        "Total Hosts",
        "Total Cities",
        "Total Neighbourhoods"
    ],
    "Value": [
        dim_listing["listing_id"].nunique(),
        dim_host["host_id"].nunique(),
        dim_listing["city"].nunique(),
        dim_listing["neighbourhood"].nunique()
    ]
})

overview

,Metric,Value
0,Total Listings,279434
1,Total Hosts,181805
2,Total Cities,10
3,Total Neighbourhoods,660


### Listing Composition Overview

In [5]:
print("Property Types :", dim_listing["property_type"].nunique())
print("Room Types     :", dim_listing["room_type"].nunique())

Property Types : 143
Room Types     : 4


### Room Type Distribution

In [6]:
room_type_dist = (
    dim_listing["room_type"]
    .value_counts()
    .reset_index()
)

room_type_dist.columns = ["room_type", "listing_count"]

room_type_dist

,room_type,listing_count
0,Entire place,181886
1,Private room,86945
2,Hotel room,5744
3,Shared room,4859


In [7]:
room_type_dist = (
    dim_listing["room_type"]
    .value_counts()
    .reset_index()
)

room_type_dist.columns = [
    "room_type",
    "listing_count"
]

room_type_dist["percentage"] = round(
    room_type_dist["listing_count"]
    / room_type_dist["listing_count"].sum()
    * 100,
    2
)

room_type_dist

,room_type,listing_count,percentage
0,Entire place,181886,65.09
1,Private room,86945,31.11
2,Hotel room,5744,2.06
3,Shared room,4859,1.74


#### Key Finding

The Airbnb market is dominated by Entire Place listings, which account for 65.09% of all listings. Private Rooms represent 31.11% of the inventory, while Hotel Rooms (2.06%) and Shared Rooms (1.74%) form a very small portion of the market.

This suggests that Airbnb primarily operates as a whole-property rental marketplace, with shared accommodation options representing a niche segment.

### Property Type Distribution

In [8]:
property_type_dist = (
    dim_listing["property_type"]
    .value_counts()
    .head(20)
    .reset_index()
)

property_type_dist.columns = [
    "property_type",
    "listing_count"
]

property_type_dist

,property_type,listing_count
0,Entire apartment,138897
1,Private room in apartment,47296
2,Private room in house,13289
3,Entire house,13261
4,Entire condominium,11245
5,Room in boutique hotel,5705
6,Entire loft,4584
7,Private room in condominium,4459
8,Private room in bed and breakfast,4238
9,Entire serviced apartment,3973


In [9]:
property_type_dist = (
    dim_listing["property_type"]
    .value_counts()
    .head(20)
    .reset_index()
)

property_type_dist.columns = [
    "property_type",
    "listing_count"
]

property_type_dist["percentage"] = round(
    property_type_dist["listing_count"]
    / len(dim_listing)
    * 100,
    2
)

property_type_dist

,property_type,listing_count,percentage
0,Entire apartment,138897,49.71
1,Private room in apartment,47296,16.93
2,Private room in house,13289,4.76
3,Entire house,13261,4.75
4,Entire condominium,11245,4.02
5,Room in boutique hotel,5705,2.04
6,Entire loft,4584,1.64
7,Private room in condominium,4459,1.60
8,Private room in bed and breakfast,4238,1.52
9,Entire serviced apartment,3973,1.42


#### Key Finding

The Airbnb market is heavily concentrated around apartment-based accommodations.

Entire Apartments alone account for 49.71% of all listings, while Private Rooms in Apartments contribute another 16.93%. Together, apartment-based inventory represents the majority of the marketplace.

Traditional accommodation types such as boutique hotels, hotels, guesthouses, villas, and aparthotels collectively form a relatively small share of listings, indicating that Airbnb primarily functions as a residential accommodation platform rather than a hotel marketplace.

### Missing Review Scores Analysis

In [10]:
review_score_missing = (
    dim_listing[
        "review_scores_rating"
    ]
    .isna()
    .sum()
)

print("Listings Missing Review Rating:")
print(review_score_missing)

print("\nPercentage:")
print(
    round(
        review_score_missing
        / len(dim_listing)
        * 100,
        2
    ),
    "%"
)

Listings Missing Review Rating:
91235

Percentage:
32.65 %


#### Key Finding

Approximately 32.65% of listings do not have a review rating.

This indicates that a substantial portion of Airbnb listings either have not received reviews or do not yet have sufficient review information available. As a result, analyses involving customer satisfaction and review scores should account for missing values and focus primarily on listings with available ratings.

### Airbnb Supply Distribution by City

In [11]:
city_listings = (
    dim_listing.groupby("city")
    .size()
    .reset_index(name="listing_count")
    .sort_values(
        by="listing_count",
        ascending=False
    )
)

city_listings["percentage"] = round(
    city_listings["listing_count"]
    / city_listings["listing_count"].sum()
    * 100,
    2
)

city_listings

,city,listing_count,percentage
6,Paris,64595,23.12
5,New York,36966,13.23
9,Sydney,33596,12.02
8,Rome,27630,9.89
7,Rio de Janeiro,26584,9.51
3,Istanbul,24503,8.77
4,Mexico City,20048,7.17
0,Bangkok,19359,6.93
1,Cape Town,19070,6.82
2,Hong Kong,7083,2.53


#### Key Finding

The Airbnb marketplace is not evenly distributed across cities.

Paris is the largest market in the dataset with 64,595 listings, representing 23.12% of all listings. New York (13.23%) and Sydney (12.02%) form the next largest markets.

At the other end of the spectrum, Hong Kong contributes only 2.53% of total listings, making it the smallest market in the dataset.

The results indicate significant variation in Airbnb supply across cities, suggesting that market size and competition levels differ substantially between locations.

### Neighbourhood Concentration by City

In [12]:
city_neighbourhoods = (
    dim_listing.groupby("city")["neighbourhood"]
    .nunique()
    .reset_index(name="neighbourhood_count")
    .sort_values(
        by="neighbourhood_count",
        ascending=False
    )
)

city_neighbourhoods

,city,neighbourhood_count
5,New York,220
7,Rio de Janeiro,151
1,Cape Town,93
0,Bangkok,50
3,Istanbul,39
9,Sydney,38
6,Paris,20
2,Hong Kong,18
4,Mexico City,16
8,Rome,15


#### Key Finding

Neighbourhood coverage varies significantly across cities.

New York has the highest geographical spread with 220 neighbourhoods, followed by Rio de Janeiro with 151 and Cape Town with 93. In contrast, Rome (15), Mexico City (16), Hong Kong (18), and Paris (20) have far fewer neighbourhood categories.

This suggests that neighbourhood definitions are not standardized across cities and may reflect differences in local administrative structures rather than actual market size. Therefore, neighbourhood counts should be interpreted cautiously when comparing cities.

### Average Price by Room Type

In [13]:
room_price = (
    dim_listing.groupby("room_type")["price"]
    .agg(["mean", "median", "count"])
    .round(2)
    .sort_values(
        by="mean",
        ascending=False
    )
)

room_price

,mean,median,count
room_type,,,
Hotel room,815.96,243.0,5744
Entire place,673.47,170.0,181886
Shared room,580.25,100.0,4859
Private room,462.48,104.0,86945


In [14]:
room_price["mean_median_gap"] = round(
    room_price["mean"] - room_price["median"],
    2
)

room_price

,mean,median,count,mean_median_gap
room_type,,,,
Hotel room,815.96,243.0,5744,572.96
Entire place,673.47,170.0,181886,503.47
Shared room,580.25,100.0,4859,480.25
Private room,462.48,104.0,86945,358.48


#### Key Finding

Room type has a significant impact on pricing. However, large differences between average and median prices indicate that the price distribution is highly skewed by expensive listings.

Hotel Rooms have the highest median price (243), followed by Entire Places (170), Private Rooms (104), and Shared Rooms (100).

Because the mean price is substantially higher than the median across all room types, median price will be used as the primary pricing metric in subsequent analyses.

### Overall Price Distribution

In [15]:
dim_listing["price"].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

count    279434.000000
mean        609.130861
std        3443.459519
min           1.000000
25%          75.000000
50%         150.000000
75%         475.000000
90%        1200.000000
95%        2000.000000
99%        6500.000000
max      625216.000000
Name: price, dtype: float64

#### Key Finding

The Airbnb price distribution is extremely right-skewed.

- Median price is 150.
- Average price is 609.
- The most expensive 1% of listings are priced above 6,500.
- The maximum listing price reaches 625,216.

The large gap between mean and median confirms the presence of substantial price outliers and luxury listings. Therefore, median price will be used as the primary metric for pricing comparisons throughout the analysis.

### Accommodation Capacity vs Price

In [16]:
capacity_price = (
    dim_listing.groupby("accommodates")["price"]
    .median()
    .reset_index(name="median_price")
    .sort_values("accommodates")
)

capacity_price.head(20)

,accommodates,median_price
0,1,80.0
1,2,110.0
2,3,150.0
3,4,175.0
4,5,225.0
5,6,337.0
6,7,384.0
7,8,600.0
8,9,550.0
9,10,997.0


#### Key Finding

Accommodation capacity shows a strong positive relationship with price.

Median price increases consistently from 80 for single-guest listings to 600 for properties accommodating 8 guests. Larger properties generally command higher prices because they can serve groups and families.

While some fluctuations appear for capacities above 8 guests, the overall trend indicates that accommodation capacity is a major driver of Airbnb pricing.

In [17]:
capacity_price["price_change"] = (
    capacity_price["median_price"]
    .diff()
)

capacity_price.head(16)

,accommodates,median_price,price_change
0,1,80.0,NaN
1,2,110.0,30.0
2,3,150.0,40.0
3,4,175.0,25.0
4,5,225.0,50.0
5,6,337.0,112.0
6,7,384.0,47.0
7,8,600.0,216.0
8,9,550.0,-50.0
9,10,997.0,447.0


#### Key Finding

The relationship between accommodation capacity and price is positive but not perfectly linear.

Median prices increase steadily from small properties (1–5 guests) to larger group accommodations. Price growth becomes much steeper for listings accommodating 6 or more guests, suggesting a premium for larger properties that serve families and groups.

The fluctuations observed beyond 8 guests are likely caused by smaller sample sizes in high-capacity listings rather than a true decline in pricing power.

Accommodation capacity appears to be one of the strongest drivers of Airbnb pricing in the dataset.

### Bedrooms vs Price

In [18]:
bedroom_price = (
    dim_listing
    .dropna(subset=["bedrooms"])
    .groupby("bedrooms")["price"]
    .median()
    .reset_index(name="median_price")
)

bedroom_price.head(20)

,bedrooms,median_price
0,1.0,117.0
1,2.0,249.0
2,3.0,410.0
3,4.0,826.5
4,5.0,1200.0
5,6.0,2000.0
6,7.0,2143.0
7,8.0,2421.5
8,9.0,900.0
9,10.0,550.0


In [19]:
bedroom_summary = (
    dim_listing
    .dropna(subset=["bedrooms"])
    .groupby("bedrooms")["price"]
    .agg(
        median_price="median",
        listing_count="count"
    )
    .reset_index()
)

bedroom_summary.head(20)

,bedrooms,median_price,listing_count
0,1.0,117.0,170069
1,2.0,249.0,51355
2,3.0,410.0,18505
3,4.0,826.5,6578
4,5.0,1200.0,2103
5,6.0,2000.0,701
6,7.0,2143.0,246
7,8.0,2421.5,124
8,9.0,900.0,79
9,10.0,550.0,150


#### Key Finding

Bedroom count exhibits one of the strongest relationships with price in the dataset.

Median price increases consistently from 117 for one-bedroom listings to 2,421.5 for eight-bedroom listings. This demonstrates that larger properties command substantial pricing premiums.

The relationship remains reliable up to approximately eight bedrooms, where sample sizes remain sufficient. Beyond eight bedrooms, listing counts become very small and pricing patterns become unstable, making those categories less suitable for general market conclusions.

Overall, bedroom count appears to be an even stronger pricing driver than accommodation capacity.

### Property Type vs Price

In [20]:
property_price = (
    dim_listing.groupby("property_type")["price"]
    .agg(
        median_price="median",
        listing_count="count"
    )
    .query("listing_count >= 1000")
    .sort_values(
        by="median_price",
        ascending=False
    )
)

property_price.head(15)

,median_price,listing_count
property_type,,
Entire villa,4487.5,1508
Entire house,650.0,13261
Private room in hostel,550.0,1232
Private room in guesthouse,508.5,1330
Entire condominium,496.0,11245
Entire serviced apartment,400.0,3973
Entire guest suite,400.0,2273
Private room in guest suite,349.0,1369
Entire townhouse,339.5,2330


#### Key Finding

Property type has a substantial impact on Airbnb pricing.

Entire Villas command the highest median price (4,487.5), significantly exceeding all other property categories. Among mainstream accommodation types, Entire Houses (650) and Entire Condominiums (496) achieve the highest median prices.

A clear pattern emerges where entire-property accommodations consistently outperform room-based accommodations in pricing. This suggests that privacy, exclusivity, and property size are major contributors to Airbnb listing value.

The results indicate that property type is one of the strongest pricing drivers in the dataset, alongside bedroom count and accommodation capacity.

### Instant Bookable vs Price

In [21]:
instant_price = (
    dim_listing.groupby("instant_bookable")["price"]
    .agg(
        median_price="median",
        listing_count="count"
    )
)

instant_price

,median_price,listing_count
instant_bookable,,
f,140.0,163882
t,180.0,115552


#### Key Finding

Instant booking appears to be associated with higher-priced listings.

Listings that support instant booking have a median price of 180, compared to 140 for listings that require host approval. This represents a pricing premium of approximately 28.6%.

The result suggests that convenience and booking flexibility may contribute to higher perceived value, allowing hosts to charge higher prices.

### Review Rating by City

In [22]:
city_ratings = (
    dim_listing.groupby("city")["review_scores_rating"]
    .agg(
        average_rating="mean",
        listing_count="count"
    )
    .round(2)
    .sort_values(
        by="average_rating",
        ascending=False
    )
)

city_ratings

,average_rating,listing_count
city,,
Mexico City,94.84,14477
Rio de Janeiro,94.57,16104
Cape Town,94.41,13426
New York,93.77,26766
Rome,93.52,20884
Sydney,93.23,22350
Paris,93.06,48013
Bangkok,93.00,11181
Istanbul,91.07,11223


#### Key Finding

Customer satisfaction is consistently high across all Airbnb markets, with average ratings exceeding 89 in every city.

Mexico City achieves the highest average rating (94.84), followed closely by Rio de Janeiro (94.57) and Cape Town (94.41). Hong Kong records the lowest average rating (89.70), although it still maintains a strong overall guest satisfaction level.

The relatively small spread between the highest and lowest ratings suggests that Airbnb hosts across all cities generally deliver positive guest experiences.

### Room Type Mix by City

In [23]:
room_city = pd.crosstab(
    dim_listing["city"],
    dim_listing["room_type"],
    normalize="index"
).round(3) * 100

room_city

room_type,Entire place,Hotel room,Private room,Shared room
city,,,,
Bangkok,54.9,5.3,36.1,3.7
Cape Town,74.2,1.6,23.6,0.7
Hong Kong,36.5,2.5,54.9,6.1
Istanbul,50.7,3.3,43.3,2.8
Mexico City,52.6,1.2,44.3,1.9
New York,52.5,0.7,45.0,1.9
Paris,85.7,2.1,11.6,0.7
Rio de Janeiro,72.5,0.3,24.9,2.3
Rome,62.4,4.6,32.3,0.7


#### Key Finding

The structure of Airbnb markets differs significantly across cities.

Paris has the highest concentration of Entire Place listings (85.7%), indicating a market focused primarily on full-property rentals. Cape Town (74.2%) and Rio de Janeiro (72.5%) also show strong dominance of entire-home accommodations.

In contrast, Hong Kong is the only city where Private Rooms (54.9%) outnumber Entire Places (36.5%), suggesting a more space-constrained and shared-accommodation market. New York, Mexico City, and Istanbul exhibit a relatively balanced mix between Entire Places and Private Rooms.

Overall, the results indicate that Airbnb serves different accommodation needs across cities, ranging from whole-property vacation rentals to room-sharing and budget-oriented stays.

### Superhost Distribution by City

In [24]:
host_city = (
    dim_listing[
        ["host_id", "city"]
    ]
    .drop_duplicates()
    .merge(
        dim_host[
            ["host_id", "host_is_superhost"]
        ],
        on="host_id",
        how="left"
    )
)

superhost_city = (
    pd.crosstab(
        host_city["city"],
        host_city["host_is_superhost"],
        normalize="index"
    ) * 100
).round(2)

superhost_city

host_is_superhost,f,t
city,,
Bangkok,85.51,14.49
Cape Town,78.85,21.15
Hong Kong,87.54,12.46
Istanbul,92.19,7.81
Mexico City,74.06,25.94
New York,82.84,17.16
Paris,88.32,11.68
Rio de Janeiro,86.34,13.66
Rome,74.37,25.63


#### Key Finding

The proportion of Superhosts varies considerably across cities.

Mexico City (25.94%) and Rome (25.63%) have the highest concentration of Superhosts, indicating a larger share of highly rated and consistently performing hosts. Cape Town also performs strongly with 21.15% Superhosts.

In contrast, Istanbul has the lowest Superhost concentration at 7.81%, followed by Sydney (10.89%) and Paris (11.68%).

These differences suggest that host quality distribution and platform maturity may vary significantly between Airbnb markets.

### Review Activity Over Time

In [25]:
reviews_by_year = (
    fact_reviews.assign(
        year=fact_reviews["date"].dt.year
    )
    .groupby("year")
    .size()
    .reset_index(name="review_count")
)

reviews_by_year

,year,review_count
0,2008,2
1,2009,115
2,2010,1236
3,2011,6253
4,2012,19922
5,2013,50522
6,2014,122132
7,2015,280332
8,2016,501754
9,2017,777198


#### Key Finding

Airbnb experienced rapid growth in review activity between 2010 and 2019.

Annual reviews increased from just 1,236 reviews in 2010 to more than 1.63 million reviews in 2019, indicating strong platform adoption and market expansion across the covered cities.

A sharp decline occurred in 2020, when review activity fell to 755,324 reviews, representing a significant disruption compared to 2019. This decline is consistent with the global impact of the COVID-19 pandemic on travel and tourism.

Review activity remains low in 2021 because the dataset only contains data through March 2021 and therefore does not represent a complete year.

### Monthly Review Seasonality

In [26]:
reviews_by_month = (
    fact_reviews.assign(
        month=fact_reviews["date"].dt.month
    )
    .groupby("month")
    .size()
    .reset_index(name="review_count")
)

reviews_by_month

,month,review_count
0,1,496797
1,2,408319
2,3,401729
3,4,375726
4,5,398159
5,6,426231
6,7,456477
7,8,441153
8,9,511226
9,10,552854


#### Key Finding

Customer review activity exhibits clear seasonal patterns throughout the year.

Review volumes are lowest between March and May, with April recording the fewest reviews (375,740). Activity gradually increases during the second half of the year, reaching its peak in October (552,877 reviews).

September and October represent the busiest review months in the dataset, while spring months show relatively lower activity. This pattern suggests stronger travel demand during late summer and autumn periods across the markets included in the dataset.

### Superhost Performance Analysis

In [27]:
superhost_performance = (
    dim_listing.merge(
        dim_host[["host_id", "host_is_superhost"]],
        on="host_id",
        how="left"
    )
    .groupby("host_is_superhost")
    .agg(
        median_price=("price", "median"),
        average_rating=("review_scores_rating", "mean"),
        listing_count=("listing_id", "count")
    )
    .round(2)
)

superhost_performance

,median_price,average_rating,listing_count
host_is_superhost,,,
f,150.0,92.26,229185
t,185.0,97.00,50249


#### Key Finding

Superhosts outperform regular hosts across both pricing and customer satisfaction metrics.

Superhost listings achieve a median price of 185 compared to 150 for non-superhosts, representing a pricing premium of approximately 23%.

Customer satisfaction differences are even more pronounced. Superhosts achieve an average rating of 97.00, substantially higher than the 92.26 average rating observed among non-superhosts.

These results suggest that Superhost status is associated with both stronger guest experiences and improved pricing performance.

### Premium Neighbourhood Analysis

In [28]:
paris_neighbourhoods = (
    dim_listing[
        dim_listing["city"] == "Paris"
    ]
    .groupby("neighbourhood")["price"]
    .agg(
        median_price="median",
        listing_count="count"
    )
    .query("listing_count >= 100")
    .sort_values(
        by="median_price",
        ascending=False
    )
)

paris_neighbourhoods.head(10)

,median_price,listing_count
neighbourhood,,
Elysee,128.0,1761
Louvre,115.0,1405
Hotel-de-Ville,109.0,1970
Luxembourg,109.0,1996
Palais-Bourbon,101.0,1763
Passy,100.0,3211
Temple,100.0,2942
Bourse,100.0,2186
Pantheon,95.0,2134


#### Key Finding – Paris

Significant pricing differences exist across Paris neighbourhoods.

Elysee commands the highest median listing price (128), followed by Louvre (115), Hotel-de-Ville (109), and Luxembourg (109). These neighbourhoods represent the premium segment of the Paris Airbnb market.

The results indicate that location within a city can have a substantial impact on pricing, even when comparing listings within the same metropolitan area.

In [29]:
premium_neighbourhoods = (
    dim_listing.groupby(
        ["city", "neighbourhood"]
    )["price"]
    .agg(
        median_price="median",
        listing_count="count"
    )
    .reset_index()
)

premium_neighbourhoods = (
    premium_neighbourhoods[
        premium_neighbourhoods["listing_count"] >= 100
    ]
)

premium_neighbourhoods = (
    premium_neighbourhoods.sort_values(
        ["city", "median_price"],
        ascending=[True, False]
    )
    .groupby("city")
    .head(5)
)

premium_neighbourhoods

,city,neighbourhood,median_price,listing_count
29,Bangkok,Parthum Wan,1857.0,573
47,Bangkok,Vadhana,1450.0,2454
8,Bangkok,Bang Rak,1379.0,1079
39,Bangkok,Samphanthawong,1341.5,144
49,Bangkok,Yan na wa,1302.0,230
103,Cape Town,Ward 54,2217.5,2552
112,Cape Town,Ward 62,2000.0,612
125,Cape Town,Ward 74,1857.0,981
119,Cape Town,Ward 69,1731.0,420
122,Cape Town,Ward 71,1725.0,302


#### Key Finding

Premium pricing is concentrated in specific neighbourhoods within each city, highlighting the importance of location as a pricing driver.

Several cities exhibit clear luxury districts. Examples include Elysee in Paris, Tribeca in New York, Miguel Hidalgo in Mexico City, Sao Conrado in Rio de Janeiro, and Pittwater in Sydney.

The results indicate that neighbourhood selection can create substantial pricing differences within the same city, making location one of the strongest determinants of Airbnb listing value alongside property size and accommodation type.

Premium neighbourhoods are generally characterized by strong tourism demand, business accessibility, waterfront locations, luxury housing markets, or proximity to major attractions.

## Exploratory Data Analysis Conclusion

The analysis revealed significant differences between Airbnb markets across the ten cities included in the dataset. Market structure, neighbourhood coverage, room-type composition, host characteristics, and customer satisfaction vary considerably across locations.

Pricing is primarily influenced by property size, bedroom count, accommodation capacity, room type, property type, and host status. Location also plays a critical role, with premium neighbourhoods consistently commanding higher prices within each city.

Review activity demonstrates strong long-term growth through 2019, followed by a sharp decline in 2020. Seasonal patterns indicate increased review activity during the second half of the year, particularly in September and October.

A limitation of the dataset is the absence of standardized currency information, preventing reliable cross-city price comparisons and value-for-travel assessments.